In [ ]:
class CreateDataframe:
    # Initialize Spark session
    spark = SparkSession.builder.appName("MultipleOrders").getOrCreate()

    # Sample data (replace with actual data source)
    data = [
        (1, "2023-01-01", "C001"),
        (2, "2023-01-02", "C002"),
        (3, "2023-01-03", "C001"),
        (4, "2023-01-04", "C003"),
        (5, "2023-01-05", "C002")
    ]

    columns = ["order_id", "order_date", "customer_id"]
    orders_df = spark.createDataFrame(data, columns)

    # Create a sample PySpark DataFrame
    data = [
        Row(name='Alice', age=30),
        Row(name='Bob', age=25),
        Row(name='Charlie', age=35)
    ]
    spark_df = spark.createDataFrame(data)

    # Sample data
    data = [
        ("Alice", 1),
        ("Bob", 2),
        ("Charlie", 3)
    ]
    # Define schema
    schema = StructType([
        StructField("Name", StringType(), True),
        StructField("ID", IntegerType(), True)
    ])
    # Create DataFrame
    df = spark.createDataFrame(data, schema)
    df.show()
    df.printSchema()


    # ✅ Fix the list: remove extra comma and use None instead of null
    data = [
        (1, 100, 2),
        (2, 50,  5),
        (1, 200, 2),
        (3, 200, None)   # None == null in Spark
    ]

    # Optional: define an explicit schema
    schema = StructType([
        StructField("order_id", IntegerType(), nullable=False),
        StructField("price",    IntegerType(), nullable=True),
        StructField("quantity", IntegerType(), nullable=True),
    ])

    df = spark.createDataFrame(data, schema)

    df.show()
    df.printSchema()


In [ ]:
class withColumn_withColumnRenamed:
    # Rename column 'name' to 'full_name'
    df_renamed = df.withColumnRenamed("name", "full")
    # Add a new column 'country' with a constant value
    df_new = df.withColumn("country", lit("India"))
    # Add a new column 'age_plus_5' which is age + 5
    df_new = df.withColumn("age_plus_5", col("age") + 5)
    # Add a new column 'age_group' based on condition
    df_with_age_group = df.withColumn(
        "age_group",
        when(col("age") >= 18, "Adult").otherwise("Minor")
    )
    # Group by 'name' and calculate average score
    # aggregation functions like sum(), count(), max(), min() from pyspark.sql.functions.
    avg_scores = df.groupBy("name").agg(avg("score").alias("average_score"))
    # Find customers with more than one order
    multi_order_customers = orders_df.groupBy("customer_id") \
        .agg(count("order_id").alias("order_count")) \
        .filter(col("order_count") > 1)


In [ ]:
# from sales table columns: sale_id, client_id, product_id, sale_date, amount, sales),
#  Which client spent the most money overall? write a pypsark code
class pyspark_GroupBy:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import sum, col
    # Initialize Spark session
    spark = SparkSession.builder.appName("TopClientBySpend").getOrCreate()
   
    client_spend_df = sales_df.groupBy("client_id").agg(sum(col("amount")).alias("total_spent"))
    # Find the client with the maximum total_spent
    top_client = client_spend_df.orderBy(col("total_spent").desc()).limit(1)
    # Show the result
    top_client.show()

In [ ]:
class collectlist_Agg:
    from pyspark.sql.functions import collect_list, col
    # Group by Col_A and collect all Col_B values into a list
    grouped_df = df.groupBy("Col_A").agg(collect_list("Col_B").alias("values"))
    # Expand list into separate columns
    final_df = grouped_df.select( col("Col_A"),
        col("values")[0].alias("Column_1"),
        col("values")[1].alias("Column_2"),
        col("values")[2].alias("Column_3")
    )
    final_df.show(truncate=False)

extract each employee’s first swipe-in time and last swipe-out time per day from a login DataFrame in PySpark

In [ ]:
# extract each employee’s first swipe-in time and last swipe-out time per day
class EmployeeSwipeTimes:
    from pyspark.sql import SparkSession
    from pyspark.sql.window import Window
    from pyspark.sql import functions as F

    # 1. Create a new column just for the Date (removing the hours/minutes)
    df_with_date = df.withColumn("date", F.to_date("timestamp"))
    # 2. Group by Employee and Date, then find the first and last swipe
    result_df = df_with_date.groupBy("employee_id", "date") \
        .agg(
            F.min("timestamp").alias("start_time"),
            F.max("timestamp").alias("end_time")
        )
    # Show the final results
    result_df.show()

    #or

    w = Window.partitionBy("emp_id", "date")

    df_with_bounds = df.withColumn(
        "start_time", F.min(F.when(F.col("swipe_type") == "IN", F.col("swipe_time"))).over(w)
    ).withColumn(
        "end_time", F.max(F.when(F.col("swipe_type") == "OUT", F.col("swipe_time"))).over(w)
    )

    #or
    df2 =df1.withColumn('work_date', to_date('date_ts').
        groupBy('emp_id','work_date').agg(
            min(when(col('status') =='login', col('date_ts'))).alias('start_date').
            max(when(col('status') =='logout', col('date_ts'))).alias('end_date')
        )
        .withColumn('duration',round(
            unix_timestamp(col('start_date'))- unix_timestamp(col('end_date'))/3600, 2)
        ))

In [ ]:
class pyspark_expr:
    #✅ What is expr()?
    # In PySpark, the expr() function (from pyspark.sql.functions) lets you write SQL expressions inside your DataFrame transformations.
    #It evaluates a SQL expression string on DataFrame columns.
    #Useful for arithmetic, conditional logic, type casting, and built-in SQL functions.
    #Equivalent to writing SQL inside PySpark code.
    from pyspark.sql.functions import expr
    expr("SQL_expression")
    df.withColumn("value_plus_5", expr("value + 5"))
    df.withColumn("category_flag", expr("CASE WHEN value > 100 THEN 'High' ELSE 'Low' END"))

+-------+-------+-----------+--------------+------------------+---------------------+
| month |country|trans_count|approved_count|trans_total_amount|approved_total_amount|
+-------+-------+-----------+--------------+------------------+---------------------+
|2018-12|    US |          2|             1|              3000|                 1000|
|2019-01|    DE |          1|             1|              2000|                 2000|
|2019-01|    US |          1|             1|              2000|                 2000|
+-------+-------+-----------+--------------+------------------+---------------------+

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, when, sum as _sum, count

# Initialize Spark session
spark = SparkSession.builder.appName("TransactionAggregation").getOrCreate()
# Sample data
data = [
    (121, "US", "approved", 1000, "2018-12-18"),
    (122, "US", "declined", 2000, "2018-12-19"),
    (123, "US", "approved", 2000, "2019-01-01"),
    (124, "DE", "approved", 2000, "2019-01-07")
]
columns = ["id", "country", "state", "amount", "trans_date"]
# Create DataFrame
df = spark.createDataFrame(data, columns)
# Transform and aggregate
result = (
    df.withColumn("month", date_format(col("trans_date"), "yyyy-MM"))
      .groupBy("month", "country")
      .agg(
          count("*").alias("trans_count"),
          _sum(when(col("state") == "approved", 1).otherwise(0)).alias("approved_count"),
          _sum(col("amount")).alias("trans_total_amount"),
          _sum(when(col("state") == "approved", col("amount")).otherwise(0)).alias("approved_total_amount")
      )
      .orderBy("month", "country")
)
result.show()


In [ ]:
class WindowFunctionOp5:
    # find the max sale from the order table from each category in new column using pyspark

    from pyspark.sql.window import Window
    from pyspark.sql.functions import max, col

    windowSpec = Window.partitionBy("category").orderBy(col('sales').desc())
    df_with_max = df.withColumn("max_sale", max("sale").over(windowSpec))

    # or

    max_sales_df = df.groupBy("category").agg(max("sale").alias("max_sale"))
    result_df = df.join(max_sales_df, on="category", how="left")

class WindowFunctionOp9:
    from pyspark.sql import SparkSession
    from pyspark.sql.window import Window
    from pyspark.sql.functions import dense_rank

    # Initialize Spark session
    spark = SparkSession.builder.appName("ThirdHighestSalary").getOrCreate()
    # Sample data
    data = [
        ("Alice", 5000),
        ("Bob", 7000),
        ("Charlie", 6000),
        ("David", 7000),
        ("Eve", 4000)
    ]
    # Create DataFrame
    columns = ["Name", "Salary"]
    df = spark.createDataFrame(data, columns)
    # Define window spec
    window_spec = Window.orderBy(df["Salary"].desc())
    # Add rank column
    ranked_df = df.withColumn("Rank", dense_rank().over(window_spec))
    # Filter for 3rd highest salary
    third_highest_salary_df = ranked_df.filter(ranked_df["Rank"] == 3)
    # Show result
    third_highest_salary_df.show()

In [ ]:
# running total or cumulative sum using window function in pyspark
class WindowFunction_RunningTotal:
    from pyspark.sql import functions as F
    from pyspark.sql import Window

    df = 'data'
    df = df.withColumn("salary_clean", F.coalesce(F.col("salary").cast("double"), F.lit(0.0)))

    windowSpec = Window.partitionBy("department")
    .orderBy(F.desc("salary"), F.asc("employee_id"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)  # robust frame


    df_with_cumulative_sum = df.withColumn("cumulative_salary",
                                           F.sum(F.col("salary"))).over(windowSpec)
                                           
class readfile:
       # 3. Read file from S3 into DataFrame
       df = spark.read \
       .format("csv") \
       .option("header", "true") \
       .load("s3a://your-bucket-name/path/to/file.csv")


       spark.read.json("s3a://your-bucket/path/file.json")
       spark.read.parquet("s3a://your-bucket/path/file.parquet")



# pivot in pyspark

pivot_df = df.groupby('name').pivot('months').agg(sum('revenue'))


In [ ]:
class pysparkRDDOp11:
    # filter() with lambda in rdd
    # Example: Filter even numbers from an RDD
    rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5, 6])
    filtered_rdd = rdd.filter(lambda x: x % 2 == 0)
    print(filtered_rdd.collect())  # Output: [2, 4, 6]


    # Using DataFrame API
    spark = SparkSession.builder.appName("FilterExample").getOrCreate()
    data = [(1, "Alice"), (2, "Bob"), (3, "Charlie")]
    df = spark.createDataFrame(data, ["id", "name"])
    # Filter rows where id > 1
    filtered_df = df.filter(col("id") > 1)
    filtered_df.show()

    # or
    filtered_rdd = df.rdd.filter(lambda row: row.id > 1)
    print(filtered_rdd.collect())

    # Filter by Multiple Conditions (RDD)
    rdd = spark.sparkContext.parallelize([("Alice", 25), ("Bob", 30), ("Charlie", 35), ("David", 40)])
    # Filter people older than 30 and name starts with 'C'
    filtered_rdd = rdd.filter(lambda x: x[1] > 30 and x[0].startswith("C"))
    print(filtered_rdd.collect())  # Output: [('Charlie', 35)]

    # ✅ 2. Filter Strings by Substring
    rdd = spark.sparkContext.parallelize(["apple", "banana", "grape", "apricot", "orange"])
    # Filter words that contain 'ap'
    filtered_rdd = rdd.filter(lambda word: "ap" in word)
    print(filtered_rdd.collect())  # Output: ['apple', 'grape', 'apricot']

    # ✅ 4. Using Lambda on DataFrame RDD
    filtered_rdd = df.rdd.filter(lambda row: row.age > 30 and row.name.startswith("C"))
    print(filtered_rdd.collect())  # Output: [Row(name='Charlie', age=35)]


In [ ]:
# Initialize Spark session
spark = SparkSession.builder.appName("WordCountDF").getOrCreate()
# Path to your folder containing txt files
folder_path = "/path/to/your/folder/*.txt"
# Read all text files into a DataFrame (each line is a row)
df = spark.read.text(folder_path)
# Split each line into words and explode into individual rows
words_df = df.select(explode(split(col("value"), "\\s+")).alias("word"))
# Group by word and count occurrences
word_counts_df = words_df.groupBy("word").count()

In [ ]:
# write a python3 program to create or generate a list of email adresses from e001@hex.com to ... e501@hex.com
lst = []
for i in range(1,502):
    format_i = str(i).zfill(3) 
    e = "e" + str(format_i) + "@hex.com"
    lst.append(e)
print(lst)

['e001@hex.com', 'e002@hex.com', 'e003@hex.com', 'e004@hex.com', 'e005@hex.com', 'e006@hex.com', 'e007@hex.com', 'e008@hex.com', 'e009@hex.com', 'e010@hex.com', 'e011@hex.com', 'e012@hex.com', 'e013@hex.com', 'e014@hex.com', 'e015@hex.com', 'e016@hex.com', 'e017@hex.com', 'e018@hex.com', 'e019@hex.com', 'e020@hex.com', 'e021@hex.com', 'e022@hex.com', 'e023@hex.com', 'e024@hex.com', 'e025@hex.com', 'e026@hex.com', 'e027@hex.com', 'e028@hex.com', 'e029@hex.com', 'e030@hex.com', 'e031@hex.com', 'e032@hex.com', 'e033@hex.com', 'e034@hex.com', 'e035@hex.com', 'e036@hex.com', 'e037@hex.com', 'e038@hex.com', 'e039@hex.com', 'e040@hex.com', 'e041@hex.com', 'e042@hex.com', 'e043@hex.com', 'e044@hex.com', 'e045@hex.com', 'e046@hex.com', 'e047@hex.com', 'e048@hex.com', 'e049@hex.com', 'e050@hex.com', 'e051@hex.com', 'e052@hex.com', 'e053@hex.com', 'e054@hex.com', 'e055@hex.com', 'e056@hex.com', 'e057@hex.com', 'e058@hex.com', 'e059@hex.com', 'e060@hex.com', 'e061@hex.com', 'e062@hex.com', 'e063@h

In [ ]:
s = "7"
print(s.rjust(3, "0"))  # "007"

s = "7"
print(s.ljust(3, "0"))  # "700"

007
